# [7.2] Feature Verbalizers - Solutions

Reference validation notebook for the section-local feature verbalizer implementation. This executes the visible tests against `solutions.py`, checks the CPU notebook contract, and verifies the committed CUDA report highlights.

Expected CUDA highlights: pinned TransformerLens `gelu-1l` loads on CUDA, a real residual direction separates safe held-out prompt labels, a concise keyword explanation beats the baseline and survives contrastives, counterexamples are zero, direction addition matches the predicted intervention, and the explanation is shorter than the examples-only baseline.


In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter7_activation_to_language"
section = "part2_feature_verbalizers"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part2_feature_verbalizers.tests as tests
from part2_feature_verbalizers import solutions


In [ ]:
tests.test_gather_verbalizer_examples_covers_top_bottom_random_contrastive(
    solutions.gather_verbalizer_examples,
)
tests.test_keyword_predictions_and_explanation_report_use_baseline_and_contrastives(
    solutions.keyword_explanation_predictions,
    solutions.explanation_prediction_report,
)
tests.test_counterexamples_and_revision_are_grounded_in_failures(
    solutions.find_counterexamples,
    solutions.revise_explanation,
)
tests.test_intervention_prediction_checks_signed_direction(
    solutions.intervention_prediction_report,
)
tests.test_explanation_brevity_compares_against_examples_only_baseline(
    solutions.explanation_brevity_report,
)
tests.test_notebook_contract(solutions.run_smoke_test)

tests.test_keyword_predictions_do_not_match_substrings(solutions.keyword_explanation_predictions)
tests.test_learned_verbalizer_terms_do_not_use_heldout_only_words(solutions.learn_verbalizer_terms)


In [ ]:
contract = solutions.run_smoke_test(cpu=True)
contract


In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
assert gpu["preflight_passed"], "7.2 CUDA preflight should pass."
assert gpu["model_name"] == "gelu-1l", "The real-model path should use pinned TransformerLens gelu-1l."
assert gpu["hf_revision"] == "bddc0e332f0ae84279e6a6a45d91b314899e1603", "gelu-1l revision should remain pinned."
assert gpu["score_separation"] >= 5.0, "Residual direction should separate positive and negative train examples."
assert gpu["prediction_accuracy"] == 1.0, "Explanation should predict held-out labels exactly in the preflight."
assert gpu["baseline_accuracy"] == 0.5, "Always-negative baseline should be recorded."
assert gpu["contrastive_accuracy"] == 1.0, "Explanation should survive contrastive examples."
assert gpu["passes_baseline"], "Explanation should beat the baseline."
assert gpu["survives_contrastive"], "Explanation should pass contrastive near-misses."
assert gpu["num_counterexamples"] == 0, "Held-out preflight should have zero counterexamples."
assert gpu["intervention_delta"] >= 0.5, "Direction intervention should move the target logit difference."
assert gpu["matches_intervention_prediction"], "Intervention should match the explanation-predicted direction."
assert gpu["brevity_shorter_than_examples"], "Explanation should be shorter than examples-only baseline."
assert gpu["train_heldout_overlap_count"] == 0, "Train and heldout prompts should be disjoint."
assert gpu["learned_terms_from_train"], "Explanation terms should be learned from train examples only."
assert gpu["target_beats_random_intervention"], "Target direction should beat the random-direction control."
assert gpu["peak_vram_gb"] <= 1.0, "The 7.2 preflight should stay under the locked 1GB budget."
{
    "score_separation": gpu["score_separation"],
    "prediction_accuracy": gpu["prediction_accuracy"],
    "contrastive_accuracy": gpu["contrastive_accuracy"],
    "intervention_delta": gpu["intervention_delta"],
    "num_counterexamples": gpu["num_counterexamples"],
    "peak_vram_gb": gpu["peak_vram_gb"],
}
